# Tutorial - SageMath - CoCalc - Interactive Simplex Method

## 1. Configuração do Ambiente

Nesta etapa, configuramos o ambiente do CoCalc para voltar a renderizar o LaTeX.

Essa célula deve ser executada apenas:
- Ao abrir o notebook pela primeira vez
- Ou quando o ambiente for reiniciado/desconectado

In [1]:
from sage.numerical.interactive_simplex_method import (
    InteractiveLPProblem, 
    InteractiveLPProblemStandardForm,
    LPDictionary,
)
from IPython.display import Latex, display
from sage.misc.html import HtmlFragment
import re

def _repr_latex_(self):
    tex = self._latex_()

    tex = tex.replace(r"\displaystyle", "")
    tex = tex.replace(r"\mspace{-6mu}", "")
    tex = tex.replace(r"\end{aligned}", r"\end{array}")
    tex = tex.replace(
        r"\end{aligned} \\",
        r"\end{array} \\"
    )

    return r"\[" + tex + r"\]"

InteractiveLPProblem._repr_latex_ = _repr_latex_

if not hasattr(InteractiveLPProblemStandardForm, "_sage_original_run_simplex_method"):
    InteractiveLPProblemStandardForm._sage_original_run_simplex_method = (
        InteractiveLPProblemStandardForm.run_simplex_method
    )

def show_dictionary_latex(d):

    tex = d._latex_()

    tex = tex.replace(
        r"\renewcommand{\arraystretch}{1.5} %notruncate",
        ""
    )
    tex = tex.replace(r"\mspace{-6mu}", "")

    return r"\[" + tex + r"\]"

def run_simplex_method_problem_latex(self):

    output = []

    d = self.initial_dictionary()

    if not d.is_feasible():

        # Substitui d._html_()
        tex = show_dictionary_latex(d)
        display(Latex(tex))

        output.append(
            "The initial dictionary is infeasible, solving auxiliary problem."
        )

        ad = self.auxiliary_problem().initial_dictionary()

        ad.enter(self.auxiliary_variable())

        ad.leave(
            min(
                zip(
                    ad.constant_terms(),
                    ad.basic_variables()
                )
            )[1]
        )

        R = ad.run_simplex_method()

        output.append(R)

        if ad.objective_value() < 0:

            output.append("The original problem is infeasible.")

            self._final_dictionary = ad

        else:

            output.append("Back to the original problem.")

            d = self.feasible_dictionary(ad)


    if d.is_feasible():

        R = d.run_simplex_method()

        output.append(R)

        if d.is_optimal():

            v = d.objective_value()

            if self._is_negative:
                v = -v

            output.append(
                ("The optimal value: ${}$. "
                 "An optimal solution: ${}$.")
                .format(
                    latex(v),
                    latex(d.basic_solution())
                )
            )

        self._final_dictionary = d

    return HtmlFragment("\n".join(map(str, output)))

InteractiveLPProblemStandardForm.run_simplex_method = (
    run_simplex_method_problem_latex
)

def _repr_latex_dictionary_(self):

    tex = self._latex_()

    tex = tex.replace(
        r"\renewcommand{\arraystretch}{1.5} %notruncate",
        ""
    )

    tex = tex.replace(
        r"\mspace{-6mu}",
        ""
    )

    return r"\[" + tex + r"\]"

LPDictionary._repr_latex_ = _repr_latex_dictionary_

if not hasattr(LPDictionary, "_sage_original_run_simplex_method"):
    LPDictionary._sage_original_run_simplex_method = (
        LPDictionary.run_simplex_method
    )

def run_simplex_method_latex(self, *args, **kwargs):

    # Chama SEMPRE o método original do Sage,
    # e não o método que foi instalado pelo patch.
    R = self._sage_original_run_simplex_method(
        *args,
        **kwargs
    )

    tex = str(R)

    # Remove equation*
    tex = tex.replace(
        r"\begin{equation*}",
        ""
    )

    tex = tex.replace(
        r"\end{equation*}",
        ""
    )

    # Remove arraystretch
    tex = tex.replace(
        r"\renewcommand{\arraystretch}{1.5} %notruncate",
        ""
    )

    # Remove mspace
    tex = tex.replace(
        r"\mspace{-6mu}",
        ""
    )

    return HtmlFragment(tex)

LPDictionary.run_simplex_method = run_simplex_method_latex

## 2. Definição do Modelo de Otimização (Exemplo da Aula 06)

In [2]:
A = ([3,2],[1,0])
b = (90,25)
c = (20,10)

In [3]:
P = InteractiveLPProblem(A, b, c, ["x_1", "x_2"], problem_type= "max", constraint_type= ["<=", "<="], variable_type= [">=", ">="])
P

LP problem (use 'view(...)' or '%display typeset' for details)

In [4]:
print("Solução Ótima:", (P.optimal_solution()))
print("Valor Ótimo:", (P.optimal_value()))

Solução Ótima: (25, 15/2)
Valor Ótimo: 575


In [5]:
Fig = P.plot(); Fig

In [6]:
Fig.save("exercicio_6-2.pdf")

In [7]:
P = P.standard_form()
P

LP problem (use 'view(...)' or '%display typeset' for details)

In [8]:
P.run_simplex_method()

\begin{array}{|rcrcrcr|}
\hline
x_{3} & = & 90 & - &\color{green}{ 3 x_{1} }& - & 2 x_{2}\\
\color{red}{x_{4} }&\color{red}{ = }&\color{red}{ 25 }&\color{red}{ - }&\color{blue}{{ x_{1} }}&\color{red}{  }&\color{red}{ }\\
\hline
z & = & 0 & + &\color{green}{ 20 x_{1} }& + & 10 x_{2}\\
\hline
\end{array}

Entering: $x_{1}$. Leaving: $x_{4}$. 


\begin{array}{|rcrcrcr|}
\hline
\color{red}{x_{3} }&\color{red}{ = }&\color{red}{ 15 }&\color{red}{ + }&\color{red}{ 3 x_{4} }&\color{red}{ - }&\color{blue}{{ 2 x_{2}}}\\
x_{1} & = & 25 & - & x_{4} &  &\color{green}{ }\\
\hline
z & = & 500 & - & 20 x_{4} & + &\color{green}{ 10 x_{2}}\\
\hline
\end{array}

Entering: $x_{2}$. Leaving: $x_{3}$. 


\begin{array}{|rcrcrcr|}
\hline
x_{2} & = & \frac{15}{2} & + & \frac{3}{2} x_{4} & - & \frac{1}{2} x_{3}\\
x_{1} & = & 25 & - & x_{4} &  & \\
\hline
z & = & 575 & - & 5 x_{4} & - & 5 x_{3}\\
\hline
\end{array}

The optimal value: $575$. An optimal solution: $\left(25,\,\frac{15}{2}\right)$.

## 3.  User-guided Interactive Simplex Method

In [9]:
A = ([3,2],[1,0])
b = (90,25)
c = (20,10)
P = InteractiveLPProblem(A, b, c, ["x_1", "x_2"], problem_type= "max", constraint_type= ["<=", "<="], variable_type= [">=", ">="])
P = P.standard_form()

In [10]:
D = P.initial_dictionary(); D

LP problem dictionary (use 'view(...)' or '%display typeset' for details)

In [11]:
D.possible_entering()

[x_1, x_2]

In [12]:
D.enter(2)

In [13]:
D.possible_leaving()

[x3]

In [14]:
D.leave(3)

In [15]:
D.update(); D

LP problem dictionary (use 'view(...)' or '%display typeset' for details)

In [16]:
D.is_optimal()

False

In [17]:
D.possible_entering()

[x_1]

In [18]:
D.enter(1)

In [19]:
D.possible_leaving()

[x4]

In [20]:
D.leave(4)

In [21]:
D.update(); D

LP problem dictionary (use 'view(...)' or '%display typeset' for details)

In [22]:
D.is_optimal()

True